# WC 2026 Knockout Predictor v2 — Data-Driven Edition

Upgraded from a purely subjective rating system to one grounded in real data:

1. **150+ years of international results (1872-present)** from the
   [martj42/international_results](https://github.com/martj42/international_results)
   dataset (auto-updated GitHub mirror of the well-known Kaggle dataset), used to
   compute an **Elo rating** for every national team.
2. The Elo system is run through **every recorded match, including this
   tournament's actual group-stage and Round-of-32 results**, so team strength
   reflects real current form, not just history.
3. **Live results lookup**: Round of 32 winners are read directly from the
   dataset where the match has been played; only matches that genuinely
   haven't happened yet are simulated. Re-running this notebook after more
   matches finish will automatically pick up the real results.
4. **Actual 2026 Golden Boot data** (as of July 3, 2026) is used to weight
   goal-scorer predictions toward players who are actually in form
   (Mbappe/Messi 6 goals, Haaland/Kane 5, etc.) rather than treating every
   squad player as equally likely to score.

**Caveat that still applies**: scorelines and exact scorer sets are
inherently high-variance. A better strength model narrows the range of
plausible outcomes, but it can't eliminate the randomness of football.

In [1]:
import csv, random, math, urllib.request

random.seed(42)  # change this to explore alternate simulated outcomes

DATA_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
urllib.request.urlretrieve(DATA_URL, "results.csv")
print("Downloaded latest results.csv")


Downloaded latest results.csv


## 1. Build Elo ratings from full match history

Standard World Football Elo methodology:
- K-factor scaled by competition importance (World Cup = 60, continental
  championships = 50, qualifiers = 40, other = 30, friendlies = 20)
- Goal-difference multiplier (bigger wins move rating more)
- Home advantage bonus of 60 points when the match isn't at a neutral venue

This produces a rating for literally every team that's played an
international match since 1872, continuously updated through this
tournament's results so far.

In [2]:
rows = []
with open("results.csv", newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        if r["home_score"] in ("NA", "") or r["away_score"] in ("NA", ""):
            continue  # not played yet
        rows.append(r)

print("Decided matches loaded:", len(rows))

elo = {}
K_BASE = {
    "FIFA World Cup": 60, "Copa America": 50, "UEFA Euro": 50,
    "African Cup of Nations": 50, "AFC Asian Cup": 50, "Confederations Cup": 45,
}

def k_for(tournament):
    if "Qualification" in tournament:
        return 40
    return K_BASE.get(tournament, 30 if "Cup" in tournament else 20)

def goal_mult(diff):
    if diff <= 1: return 1.0
    if diff == 2: return 1.5
    return (11 + diff) / 8

for r in rows:
    h, a = r["home_team"], r["away_team"]
    hs, as_ = int(r["home_score"]), int(r["away_score"])
    neutral = r["neutral"].strip().upper() == "TRUE"
    elo.setdefault(h, 1500.0)
    elo.setdefault(a, 1500.0)
    adv = 0 if neutral else 60
    dr = (elo[h] + adv) - elo[a]
    we_h = 1 / (10 ** (-dr / 400) + 1)
    w_h = 1.0 if hs > as_ else (0.0 if hs < as_ else 0.5)
    delta = k_for(r["tournament"]) * goal_mult(abs(hs - as_)) * (w_h - we_h)
    elo[h] += delta
    elo[a] -= delta

print("Elo ratings built for", len(elo), "teams")


Decided matches loaded: 49490
Elo ratings built for 336 teams


## 2. Map our bracket teams to the dataset's naming and pull current ratings

In [3]:
# code -> full name as used in the results.csv dataset
names = {
    "CAN":"Canada","MAR":"Morocco","PAR":"Paraguay","FRA":"France","BRA":"Brazil","NOR":"Norway",
    "MEX":"Mexico","ENG":"England","BEL":"Belgium","SEN":"Senegal","USA":"United States","BIH":"Bosnia and Herzegovina",
    "ESP":"Spain","AUT":"Austria","SUI":"Switzerland","ALG":"Algeria","POR":"Portugal","CRO":"Croatia",
    "ARG":"Argentina","CPV":"Cape Verde","AUS":"Australia","EGY":"Egypt","COL":"Colombia","GHA":"Ghana",
}

ratings = {code: elo.get(full, 1500.0) for code, full in names.items()}

for code, rating in sorted(ratings.items(), key=lambda x: -x[1]):
    print(f"{code:4s} {names[code]:25s} {rating:.1f}")


ARG  Argentina                 2179.6
ESP  Spain                     2177.6
FRA  France                    2157.9
BRA  Brazil                    2085.5
ENG  England                   2074.6
COL  Colombia                  2043.5
POR  Portugal                  2032.5
MEX  Mexico                    2010.7
MAR  Morocco                   2005.4
SUI  Switzerland               2004.9
BEL  Belgium                   1977.9
NOR  Norway                    1949.2
CRO  Croatia                   1909.5
USA  United States             1895.7
AUT  Austria                   1869.8
AUS  Australia                 1867.3
PAR  Paraguay                  1861.3
SEN  Senegal                   1852.9
CAN  Canada                    1847.6
EGY  Egypt                     1841.9
ALG  Algeria                   1833.8
CPV  Cape Verde                1690.0
BIH  Bosnia and Herzegovina    1675.5
GHA  Ghana                     1665.1


## 3. Key scorers, weighted by actual 2026 Golden Boot form

Jersey numbers plus a `weight` reflecting real in-tournament output as of
July 3, 2026 (e.g. Mbappe/Messi 6 goals, Haaland/Kane 5, Dembele hat-trick
vs Norway, Oyarzabal 4). Higher weight = more likely to be sampled as a
scorer. Teams without specific reported tournament form use flat weights
across their known key attackers.

In [4]:
# (jersey, name, weight)
scorers = {
    "ARG": [(10,"Messi",6.0),(9,"J.Alvarez",1.5),(22,"Lautaro",1.5)],
    "FRA": [(10,"Mbappe",6.0),(11,"Dembele",3.5),(9,"Kolo Muani",1.0)],
    "BRA": [(7,"Vinicius Jr",2.0),(11,"Cunha",2.5),(9,"Richarlison",1.0)],
    "ENG": [(9,"Kane",5.0),(7,"Saka",2.0),(10,"Foden",1.0)],
    "ESP": [(19,"Oyarzabal",4.0),(10,"Yamal",2.0),(9,"Morata",1.0)],
    "POR": [(7,"Ronaldo",3.0),(11,"B.Fernandes",1.5),(21,"Leao",1.0)],
    "NOR": [(9,"Haaland",5.0),(11,"Sorloth",1.0),(7,"Odegaard",1.0)],
    "MEX": [(7,"Quinones",3.0),(9,"R.Jimenez",1.5),(20,"Gimenez",1.0)],
    "MAR": [(9,"Saibari",3.0),(19,"En-Nesyri",2.0),(2,"Hakimi",1.0)],
    "BEL": [(9,"Lukaku",1.5),(7,"De Bruyne",1.5),(11,"Doku",1.0)],
    "USA": [(10,"Pulisic",1.5),(9,"Balogun",1.0),(11,"Weah",1.0)],
    "CRO": [(10,"Modric",1.0),(9,"Kramaric",1.0),(17,"Budimir",1.0)],
    "COL": [(7,"Diaz",2.5),(11,"Cordoba",1.0),(20,"J.Rodriguez",1.0)],
    "SUI": [(10,"Xhaka",1.0),(9,"Embolo",1.5),(7,"Ndoye",1.0)],
    "EGY": [(7,"M.Salah",2.5),(9,"Zizo",1.0),(20,"Marmoush",1.0)],
    "PAR": [(9,"Alderete",1.0),(11,"Almiron",1.0),(7,"Enciso",1.0)],
    "CAN": [(17,"David",1.5),(10,"Buchanan",1.0),(23,"Larin",1.0)],
    "AUT": [(9,"Gregoritsch",1.0),(10,"Sabitzer",1.0),(19,"Schmid",1.0)],
    "ALG": [(19,"Mahrez",1.0),(9,"Delort",1.0),(11,"Bounedjah",1.0)],
    "AUS": [(9,"Duke",1.0),(19,"Irvine",1.0),(7,"Metcalfe",1.0)],
    "CPV": [(9,"W.Semedo",1.0),(10,"R.Mendes",1.0),(7,"Stopira",1.0)],
    "GHA": [(10,"Kudus",1.5),(9,"Semenyo",1.5),(19,"Ayew",1.0)],
    "BIH": [(17,"Dzeko",1.5),(10,"Pjanic Jr",1.0),(9,"Demirovic",1.0)],
}


## 4. Poisson goal-simulation engine, calibrated on Elo gap

In [5]:
def poisson(lam):
    L = pow(2.71828182846, -lam)
    k, p = 0, 1
    while True:
        k += 1
        p *= random.random()
        if p <= L:
            return k - 1

def sim_score(a, b):
    dr = ratings[a] - ratings[b]  # neutral venue for knockout matches
    lam_a = max(0.35, 1.25 + dr / 200)
    lam_b = max(0.35, 1.25 - dr / 200)
    ga = min(6, poisson(lam_a))
    gb = min(6, poisson(lam_b))
    if ga == gb:
        if ratings[a] >= ratings[b]:
            ga += 1
        else:
            gb += 1
    return ga, gb

def weighted_sample_no_replace(pool, k):
    pool = pool[:]
    chosen = []
    for _ in range(min(k, len(pool))):
        total = sum(w for _, _, w in pool)
        r = random.uniform(0, total)
        upto = 0
        for i, (num, nm, w) in enumerate(pool):
            upto += w
            if upto >= r:
                chosen.append(pool.pop(i))
                break
    return chosen

def pick_scorers(team, goals):
    pool = scorers.get(team, [(9,"Fwd",1.0),(10,"Mid",1.0),(7,"Wing",1.0)])
    chosen = weighted_sample_no_replace(pool, goals)
    nums = [str(n) for n, _, _ in chosen]
    while len(nums) < goals:
        nums.append(str(pool[0][0]))  # brace for top-weighted scorer
    return ";".join(nums) if goals > 0 else ""

def play(a, b):
    ga, gb = sim_score(a, b)
    winner = a if ga > gb else b
    return ga, gb, winner


## 5. Bracket structure with live result lookup

For each Round-of-32 fixture still needed to seed the Round of 16, look up
the actual result in the freshly-downloaded dataset. If it's been played,
use the real winner. If not, simulate it. This means re-running this
notebook later today, after the remaining matches finish, will
automatically switch those from simulated to real without any manual
editing.

In [6]:
def real_result(team_a, team_b):
    """Look up a real match result between two teams from the 2026 World Cup
    in the dataset. Returns winner code, or None if not played yet / not found."""
    na, nb = names[team_a], names[team_b]
    for r in rows:
        if r["tournament"] != "FIFA World Cup" or not r["date"].startswith("2026"):
            continue
        h, a_ = r["home_team"], r["away_team"]
        if {h, a_} == {na, nb}:
            hs, as_ = int(r["home_score"]), int(r["away_score"])
            if hs == as_:
                continue  # shouldn't happen in knockout, skip just in case
            winner_name = h if hs > as_ else a_
            return team_a if winner_name == na else team_b
    return None

r16_pairs_raw = [
    ("CAN","MAR"),
    ("FRA","PAR"),
    ("BRA","NOR"),
    ("MEX","ENG"),
]

r32_pairs = [
    ("BEL","SEN"), ("USA","BIH"),
    ("POR","CRO"), ("ESP","AUT"),
    ("ARG","CPV"), ("AUS","EGY"),
    ("SUI","ALG"), ("COL","GHA"),
]

winners = {}
for pair in r32_pairs:
    real = real_result(*pair)
    if real is not None:
        winners[pair] = real
        print(f"{names[pair[0]]} vs {names[pair[1]]}: REAL result -> {names[real]}")
    else:
        _, _, w = play(*pair)
        winners[pair] = w
        print(f"{names[pair[0]]} vs {names[pair[1]]}: not played yet -> SIMULATED -> {names[w]}")

r16_pairs_raw += [
    (winners[("BEL","SEN")], winners[("USA","BIH")]),
    (winners[("POR","CRO")], winners[("ESP","AUT")]),
    (winners[("ARG","CPV")], winners[("AUS","EGY")]),
    (winners[("SUI","ALG")], winners[("COL","GHA")]),
]


Belgium vs Senegal: REAL result -> Belgium
United States vs Bosnia and Herzegovina: REAL result -> United States
Portugal vs Croatia: REAL result -> Portugal
Spain vs Austria: REAL result -> Spain
Argentina vs Cape Verde: not played yet -> SIMULATED -> Argentina
Australia vs Egypt: not played yet -> SIMULATED -> Egypt
Switzerland vs Algeria: REAL result -> Switzerland
Colombia vs Ghana: not played yet -> SIMULATED -> Colombia


In [7]:
results = []

def add(mid, stage, a, b):
    ga, gb, w = play(a, b)
    results.append({
        "match_id": mid, "stage": stage, "home": a, "away": b,
        "hs": ga, "as_": gb, "winner": w,
        "hsc": pick_scorers(a, ga), "asc": pick_scorers(b, gb)
    })
    return w

r16_winners = []
for i, (a, b) in enumerate(r16_pairs_raw, start=1):
    real = real_result(a, b)
    if real is not None:
        r16_winners.append(real)
        # still record a row so the notebook's own printout stays informative,
        # using a real 0-0 placeholder score isn't meaningful, so pull the real score
        for r in rows:
            if r["tournament"] == "FIFA World Cup" and r["date"].startswith("2026"):
                h, a_ = r["home_team"], r["away_team"]
                if {h, a_} == {names[a], names[b]}:
                    hs, as_ = int(r["home_score"]), int(r["away_score"])
                    if h == names[a]:
                        results.append({"match_id": f"R16_{i:03d}", "stage": "Round of 16",
                                         "home": a, "away": b, "hs": hs, "as_": as_, "winner": real,
                                         "hsc": pick_scorers(a, hs), "asc": pick_scorers(b, as_)})
                    else:
                        results.append({"match_id": f"R16_{i:03d}", "stage": "Round of 16",
                                         "home": a, "away": b, "hs": as_, "as_": hs, "winner": real,
                                         "hsc": pick_scorers(a, as_), "asc": pick_scorers(b, hs)})
                    break
        print(f"R16_{i:03d} {names[a]} vs {names[b]}: REAL result -> {names[real]}")
    else:
        w = add(f"R16_{i:03d}", "Round of 16", a, b)
        r16_winners.append(w)
        print(f"R16_{i:03d} {names[a]} vs {names[b]}: not played yet -> SIMULATED -> {names[w]}")

# QF0: left-column top half (CAN/MAR + FRA/PAR)
# QF1: right-column top half (BRA/NOR + MEX/ENG)
# QF2: left-column bottom half (BEL-SEN/USA-BIH + POR-CRO/ESP-AUT)
# QF3: right-column bottom half (ARG-CPV/AUS-EGY + SUI-ALG/COL-GHA)
qf_pairs = [(r16_winners[0],r16_winners[1]), (r16_winners[2],r16_winners[3]),
            (r16_winners[4],r16_winners[5]), (r16_winners[6],r16_winners[7])]
qf_winners, qf_losers = [], []
for i,(a,b) in enumerate(qf_pairs, start=1):
    w = add(f"QF_{i:03d}", "Quarter Final", a, b)
    qf_winners.append(w)
    r = results[-1]
    qf_losers.append(r["home"] if r["winner"]==r["away"] else r["away"])

# SF0: left column overall (QF0 winner vs QF2 winner) -- NOT QF0 vs QF1, which
#      would incorrectly cross the top-half of the left column with the top-half
#      of the right column. The bracket only crosses columns at the Final.
# SF1: right column overall (QF1 winner vs QF3 winner)
sf_pairs = [(qf_winners[0],qf_winners[2]), (qf_winners[1],qf_winners[3])]
sf_winners, sf_losers = [], []
for i,(a,b) in enumerate(sf_pairs, start=1):
    w = add(f"SF_{i:03d}", "Semi Final", a, b)
    sf_winners.append(w)
    r = results[-1]
    sf_losers.append(r["home"] if r["winner"]==r["away"] else r["away"])

add("TP_001", "Third Place Play-off", sf_losers[0], sf_losers[1])
champion = add("F_001", "Final", sf_winners[0], sf_winners[1])

print("\nPredicted champion:", names[champion])

with open("mufifa26_predictions.csv", "w", newline="") as f:
    wtr = csv.writer(f)
    wtr.writerow(["match_id","stage","home_team","away_team","predicted_home_score",
                  "predicted_away_score","predicted_scorers_home","predicted_scorers_away",
                  "predicted_winner"])
    # Template now only requests QF onward (R16 predictions window has closed)
    for r in results:
        if r["stage"] == "Round of 16":
            continue
        wtr.writerow([r["match_id"], r["stage"], names[r["home"]], names[r["away"]],
                      r["hs"], r["as_"], r["hsc"], r["asc"], names[r["winner"]]])

for r in results:
    print(r["match_id"], names[r["home"]], r["hs"], "-", r["as_"], names[r["away"]], "| Winner:", names[r["winner"]])


R16_001 Canada vs Morocco: not played yet -> SIMULATED -> Morocco
R16_002 France vs Paraguay: not played yet -> SIMULATED -> France
R16_003 Brazil vs Norway: not played yet -> SIMULATED -> Brazil
R16_004 Mexico vs England: not played yet -> SIMULATED -> England
R16_005 Belgium vs United States: not played yet -> SIMULATED -> Belgium
R16_006 Portugal vs Spain: not played yet -> SIMULATED -> Spain
R16_007 Argentina vs Egypt: not played yet -> SIMULATED -> Argentina
R16_008 Switzerland vs Colombia: not played yet -> SIMULATED -> Colombia

Predicted champion: Argentina
R16_001 Canada 0 - 1 Morocco | Winner: Morocco
R16_002 France 3 - 1 Paraguay | Winner: France
R16_003 Brazil 2 - 0 Norway | Winner: Brazil
R16_004 Mexico 2 - 3 England | Winner: England
R16_005 Belgium 1 - 0 United States | Winner: Belgium
R16_006 Portugal 0 - 1 Spain | Winner: Spain
R16_007 Argentina 3 - 0 Egypt | Winner: Argentina
R16_008 Switzerland 0 - 1 Colombia | Winner: Colombia
QF_001 Morocco 2 - 3 France | Winner: F